# totalVI MALT CITE-seq analysis

This notebook loads a 10x Genomics CITE-seq dataset and prepares it for analysis with totalVI.

## 1. Import libraries

In [1]:
import os
import tempfile
import requests

import numpy as np
import pandas as pd
import scipy

import matplotlib.pyplot as plt
import seaborn as sns

import scanpy as sc
import muon
import scvi
import torch
import anndata as ad
import mudata as md

/home/dags/miniconda3/envs/totalvi_env/lib/python3.10/site-packages/muon/_core/preproc.py:31: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  if Version(scanpy.__version__) < Version("1.10"):


## 2. Check package versions

In [2]:
print("scanpy:", sc.__version__)
print("scvi-tools:", scvi.__version__)
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

scanpy: 1.11.5
scvi-tools: 1.3.3
torch: 2.12.0+cu130
CUDA available: True


/tmp/ipykernel_29339/203591720.py:1: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  print("scanpy:", sc.__version__)


## 3. Load 10x CITE-seq data

In [3]:
T1_data = muon.read_10x_mtx(
    "../data/raw/T1/CITE-seq_pbmc_10k.untar/filtered_feature_bc_matrix",
    var_names="gene_symbols"
)

T2_data = muon.read_10x_mtx(
    "../data/raw/T2/CITE-seq_pbmc_5k.untar/filtered_feature_bc_matrix",
    var_names="gene_symbols"
)


/home/dags/miniconda3/envs/totalvi_env/lib/python3.10/site-packages/mudata/_core/mudata.py:1416: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/home/dags/miniconda3/envs/totalvi_env/lib/python3.10/site-packages/mudata/_core/mudata.py:1272: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("obs", axis=1, join_common=join_common)
/home/dags/miniconda3/envs/totalvi_env/lib/python3.10/site-packages/mudata/_core/mudata.py:1416: Fut

## 4. Check feature types

In [4]:
# Check loaded MuData objects
print("T1:", T1_data)
print("T1 modalities:", T1_data.mod.keys())

print("T2:", T2_data)
print("T2 modalities:", T2_data.mod.keys())

T1: MuData object with n_obs × n_vars = 7865 × 33555
  var:	'gene_ids', 'feature_types'
  2 modalities
    rna:	7865 × 33538
      var:	'gene_ids', 'feature_types'
    prot:	7865 × 17
      var:	'gene_ids', 'feature_types'
T1 modalities: dict_keys(['rna', 'prot'])
T2: MuData object with n_obs × n_vars = 5247 × 33570
  var:	'gene_ids', 'feature_types'
  2 modalities
    rna:	5247 × 33538
      var:	'gene_ids', 'feature_types'
    prot:	5247 × 32
      var:	'gene_ids', 'feature_types'
T2 modalities: dict_keys(['rna', 'prot'])


In [5]:
# Check RNA and protein dimensions
print("T1 RNA:", T1_data.mod["rna"].shape)
print("T1 protein:", T1_data.mod["prot"].shape)

print("T2 RNA:", T2_data.mod["rna"].shape)
print("T2 protein:", T2_data.mod["prot"].shape)

T1 RNA: (7865, 33538)
T1 protein: (7865, 17)
T2 RNA: (5247, 33538)
T2 protein: (5247, 32)


In [6]:
print("T1 Proteins list:", T1_data.mod["prot"].var_names.tolist())

print("T2 Proteins list:", T2_data.mod["prot"].var_names.tolist())

T1 Proteins list: ['CD3_TotalSeqB', 'CD4_TotalSeqB', 'CD8a_TotalSeqB', 'CD14_TotalSeqB', 'CD15_TotalSeqB', 'CD16_TotalSeqB', 'CD56_TotalSeqB', 'CD19_TotalSeqB', 'CD25_TotalSeqB', 'CD45RA_TotalSeqB', 'CD45RO_TotalSeqB', 'PD-1_TotalSeqB', 'TIGIT_TotalSeqB', 'CD127_TotalSeqB', 'IgG2a_control_TotalSeqB', 'IgG1_control_TotalSeqB', 'IgG2b_control_TotalSeqB']
T2 Proteins list: ['CD3_TotalSeqB', 'CD4_TotalSeqB', 'CD8a_TotalSeqB', 'CD11b_TotalSeqB', 'CD14_TotalSeqB', 'CD15_TotalSeqB', 'CD16_TotalSeqB', 'CD19_TotalSeqB', 'CD20_TotalSeqB', 'CD25_TotalSeqB', 'CD27_TotalSeqB', 'CD28_TotalSeqB', 'CD34_TotalSeqB', 'CD45RA_TotalSeqB', 'CD45RO_TotalSeqB', 'CD56_TotalSeqB', 'CD62L_TotalSeqB', 'CD69_TotalSeqB', 'CD80_TotalSeqB', 'CD86_TotalSeqB', 'CD127_TotalSeqB', 'CD137_TotalSeqB', 'CD197_TotalSeqB', 'CD274_TotalSeqB', 'CD278_TotalSeqB', 'CD335_TotalSeqB', 'PD-1_TotalSeqB', 'HLA-DR_TotalSeqB', 'TIGIT_TotalSeqB', 'IgG1_control_TotalSeqB', 'IgG2a_control_TotalSeqB', 'IgG2b_control_TotalSeqB']


## 5. Add batch information

In [7]:
T1_data.mod["rna"].obs["batch"] = "10kpbmc"
T2_data.mod["rna"].obs["batch"] = "5kpbmc"

## 6. Merge active samples into a single MuData object

In [8]:
# Make variable names unique
T1_data.mod["rna"].var_names_make_unique()
T2_data.mod["rna"].var_names_make_unique()

# Filter to shared genes and proteins between the two datasets with an inner join
rna_c = ad.concat([T1_data.mod["rna"], T2_data.mod["rna"]])
rna_c.obs_names_make_unique()

prot_c = ad.concat([T1_data.mod["prot"], T2_data.mod["prot"]])
prot_c.obs_names_make_unique()

mdata = md.MuData({"rna": rna_c, "prot": prot_c})

mdata

/home/dags/miniconda3/envs/totalvi_env/lib/python3.10/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/dags/miniconda3/envs/totalvi_env/lib/python3.10/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/dags/miniconda3/envs/totalvi_env/lib/python3.10/site-packages/mudata/_core/mudata.py:1416: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/home/dags/miniconda3/envs/totalvi_env/lib/python3.10/site-packages/mudata/_core

MuData object with n_obs × n_vars = 13112 × 33555
  2 modalities
    rna:	13112 × 33538
      obs:	'batch'
    prot:	13112 × 17

## 7. RNA preprocessing and highly variable gene selection

In [9]:
def filter_pbmc(mdata):
    # Store raw RNA counts
    mdata.mod["rna"].layers["counts"] = mdata.mod["rna"].X.copy()

    # Normalize and log-transform RNA counts
    sc.pp.normalize_total(mdata.mod["rna"])
    sc.pp.log1p(mdata.mod["rna"])

    # Select highly variable genes
    sc.pp.highly_variable_genes(
        mdata.mod["rna"],
        n_top_genes=4000,
        flavor="seurat_v3",
        layer="counts",
    )

    # Place subsetted counts in a new modality
    mdata.mod["rna_subset"] = mdata.mod["rna"][
        :, mdata.mod["rna"].var["highly_variable"]
    ].copy()

    mdata = md.MuData(mdata.mod)

    return mdata


mdata = filter_pbmc(mdata)

mdata

/home/dags/miniconda3/envs/totalvi_env/lib/python3.10/site-packages/mudata/_core/mudata.py:1416: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/home/dags/miniconda3/envs/totalvi_env/lib/python3.10/site-packages/mudata/_core/mudata.py:565: UserWarning: Cannot join columns with the same name because var_names are intersecting.
  self._update_attr_legacy(attr, axis, join_common, **kwargs)
/home/dags/miniconda3/envs/totalvi_env/lib/python3.10/site-packages/mudata/_core/mudata.py:1272: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the d

MuData object with n_obs × n_vars = 13112 × 37555
  3 modalities
    rna:	13112 × 33538
      obs:	'batch'
      var:	'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm'
      uns:	'log1p', 'hvg'
      layers:	'counts'
    prot:	13112 × 17
    rna_subset:	13112 × 4000
      obs:	'batch'
      var:	'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm'
      uns:	'log1p', 'hvg'
      layers:	'counts'

## 8. Save preprocessed MuData object

In [11]:
output_path = "../data/processed/citeseq_longitudinal_preprocessed.h5mu"

mdata.write(output_path)

print(f"Preprocessed MuData object saved to: {output_path}")

Preprocessed MuData object saved to: ../data/processed/citeseq_longitudinal_preprocessed.h5mu


/home/dags/miniconda3/envs/totalvi_env/lib/python3.10/site-packages/mudata/_core/mudata.py:1416: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/home/dags/miniconda3/envs/totalvi_env/lib/python3.10/site-packages/mudata/_core/mudata.py:1272: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("obs", axis=1, join_common=join_common)


In [12]:
!ls -lh ../data/processed/

total 425M
drwxr-xr-x 2 dags dags 4.0K May 27 10:44 T1
drwxr-xr-x 2 dags dags 4.0K May 27 10:44 T2
drwxr-xr-x 2 dags dags 4.0K May 27 10:44 T3
drwxr-xr-x 2 dags dags 4.0K May 27 10:44 T4
-rw-r--r-- 1 dags dags 425M May 27 12:37 citeseq_longitudinal_preprocessed.h5mu
